# 04. Scoreboard (Image Cell)

회귀 cell의 04 노트북과 동일 분석 (DSC vs accuracy):
- 산점도, 라인, 히트맵, 박스플롯
- Pearson r, Spearman ρ, 비선형 RF 5-fold R²
- Polluter hold-out, 모델별 r
- 검증 기준: r ≥ 0.4

---

In [ ]:
# ============================================================
# 0. 환경 + 데이터 로드
# ============================================================
from google.colab import drive; drive.mount('/content/drive')
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
CHARTS_DIR = f'{RESULTS_DIR}/charts_image'
os.makedirs(CHARTS_DIR, exist_ok=True)
if BASE not in sys.path: sys.path.insert(0, BASE)

dsc = pd.read_csv(f'{RESULTS_DIR}/dsc_scores_image.csv')
perf = pd.read_csv(f'{RESULTS_DIR}/model_performance_image.csv')
print(f'DSC: {len(dsc)}, perf: {len(perf)}')

merged = perf.merge(dsc[['dataset','polluter','level','score','grade']],
                    on=['dataset','polluter','level'])
merged = merged.rename(columns={'score': 'dsc_score'})
print(f'merged: {len(merged)}')

In [ ]:
# ============================================================
# 1. 산점도 + 라인 + 박스플롯
# ============================================================
plt.figure(figsize=(10, 6))
for m, sub in merged.groupby('model'):
    plt.scatter(sub['dsc_score'], sub['accuracy'], label=m, alpha=0.6, s=30)
plt.xlabel('DSC Score (image cell)'); plt.ylabel('Accuracy')
plt.title('DSC ↔ accuracy 산점도 (image cell)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig(f'{CHARTS_DIR}/01_scatter.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=merged, x='grade', y='accuracy', order=['A','B','C','D'], palette='RdYlGn_r')
plt.title('DSC 등급별 accuracy (image cell)')
plt.savefig(f'{CHARTS_DIR}/02_grade_box.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 2. 통계 검증 + 가설 판정
# ============================================================
x = merged['dsc_score'].values; y = merged['accuracy'].values
r_p, p_p = pearsonr(x, y); r_s, p_s = spearmanr(x, y)
print(f'Pearson r = {r_p:+.4f}, Spearman ρ = {r_s:+.4f}')

# 비선형
X = merged[['dsc_score']].values; rf_folds = []
for tr, te in KFold(5, shuffle=True, random_state=42).split(X):
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X[tr], y[tr])
    rf_folds.append(r2_score(y[te], rf.predict(X[te])))
print(f'비선형 RF 5-fold R² = {np.mean(rf_folds):.4f} ± {np.std(rf_folds):.4f}')

# 모델별
print('\n모델별 r:')
for m, sub in merged.groupby('model'):
    rr, pp = pearsonr(sub['dsc_score'], sub['accuracy'])
    print(f'  {m:<22s} r={rr:+.4f} p={pp:.2e} n={len(sub)}')

# polluter hold-out
print('\nPolluter hold-out:')
hold_pass = 0; n_pol = 0
for hp in sorted(merged['polluter'].unique()):
    if hp == 'none': continue
    n_pol += 1
    sub = merged[merged.polluter != hp]
    rr, _ = pearsonr(sub['dsc_score'], sub['accuracy'])
    pass_ = rr >= 0.4; hold_pass += int(pass_)
    print(f'  {hp:<22s} r={rr:+.4f}  {"PASS" if pass_ else "FAIL"}')

# 가설 판정
verdict = {
    'H1 r ≥ 0.4': r_p >= 0.4,
    'H2 ρ ≥ 0.4': r_s >= 0.4,
    'H3 비선형 우위': np.mean(rf_folds) > r_p**2,
    'H4 모든 모델 양의 r': all(pearsonr(s['dsc_score'], s['accuracy'])[0] > 0
                              for _, s in merged.groupby('model')),
    'H5 polluter hold-out ≥4/5': hold_pass >= 4,
}
print('\n=== 가설 판정 ===')
for k, v in verdict.items():
    print(f'  {"✅" if v else "❌"} {k}')
n_pass = sum(verdict.values())
print(f'\n종합: {n_pass}/5 PASS')
if n_pass == 5:
    print('🎉 image cell Phase 2 통과')